# Attribute Filtering Workflow

This notebook compares extinction-value filtering and direct attribute-threshold filtering on max-trees, min-trees, and tree-of-shapes representations.


## 1. Install the library

This cell is optional when `mmcfilters` is already installed in the active Python environment.


In [ ]:
#!pip install mmcfilters
#!pip install bokeh
#!pip install morphotreeviz

## 2. Import the library and define plotting helpers

The helper functions keep the visualization code compact so the notebook can focus on the filtering operations.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (16,16)
import cv2 as cv


def load_grayscale(path):
    image = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return np.ascontiguousarray(image, dtype=np.uint8)
import mmcfilters
import mtviz as viz
if not hasattr(viz, "show_tree"):
    viz.show_tree = lambda *args, **kwargs: None

def print_tree_with_attribute(attribute_type, attribute_by_node):
    return None
show_component_tree = getattr(viz, "show_component_tree", lambda *args, **kwargs: None)

from bokeh.io import output_notebook, show
from bokeh.layouts import column, row
output_notebook()


## 3. Create morphological trees from an input image

The image is converted into max-tree, min-tree, and tree-of-shapes representations. The adjacency radius defines the neighborhood used to connect pixels during tree construction.


In [ ]:
input_image = load_grayscale("../dat/imgObjetos.png")
#input_image = load_grayscale("../dat/imgTeste.png")

(num_rows, num_cols) = input_image.shape

max_tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(input_image, radius=1.5)
min_tree = mmcfilters.MorphologicalTreeFactory.createMinTree(input_image, radius=1.5)
tos = mmcfilters.MorphologicalTreeFactory.createTreeOfShapes(input_image, interpolation=mmcfilters.ToSInterpolation.SelfDual)
tos_4c8c = mmcfilters.MorphologicalTreeFactory.createTreeOfShapes(input_image, interpolation=mmcfilters.ToSInterpolation.Min4cMax8c)

## 4. Filter by extinction values

Extinction values rank meaningful extrema by persistence. The filter keeps the most persistent structures and removes less relevant components.


In [ ]:
max_tree_filter = mmcfilters.AttributeFilters(max_tree)
min_tree_filter = mmcfilters.AttributeFilters(min_tree)
tos_filter = mmcfilters.AttributeFilters(tos)
tos_4c8c_filter = mmcfilters.AttributeFilters(tos_4c8c)
num_leaves = 6
attribute_type = mmcfilters.Attribute.AREA

max_tree_attribute = mmcfilters.Attribute.computeSingleAttribute(max_tree, attribute_type)
min_tree_attribute = mmcfilters.Attribute.computeSingleAttribute(min_tree, attribute_type)
tos_attribute = mmcfilters.Attribute.computeSingleAttribute(tos, attribute_type)
tos_4c8c_attribute = mmcfilters.Attribute.computeSingleAttribute(tos_4c8c, attribute_type)



plt.subplot(1,5, 1)
plt.imshow(input_image, cmap='gray')
plt.title('input')

plt.subplot(1,5, 2)
plt.imshow(max_tree_filter.filteringByExtinction(max_tree_attribute, num_leaves), cmap='gray')
plt.title('reconstruction (max_tree)')

plt.subplot(1,5, 3)
plt.imshow(min_tree_filter.filteringByExtinction(min_tree_attribute, num_leaves), cmap='gray')
plt.title('reconstruction (min_tree)')

plt.subplot(1,5, 4)
plt.imshow(tos_filter.filteringByExtinction(tos_attribute, num_leaves), cmap='gray')
plt.title('reconstruction (tos)')

plt.subplot(1,5, 5)
plt.imshow(tos_4c8c_filter.filteringByExtinction(tos_4c8c_attribute, num_leaves), cmap='gray')
plt.title('reconstruction (tos_4c8c)')

### Build saliency maps from extinction values

A saliency map projects the selected extinction information back to the image domain, making it easier to compare max-tree, min-tree, and tree-of-shapes responses visually.


In [ ]:
plt.subplot(1,5, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,5, 2)
plt.imshow(max_tree_filter.saliencyMapByExtinction(max_tree_attribute, num_leaves), cmap='Grays', vmax=10000, vmin=0)
plt.title('reconstruction (max_tree)')

plt.subplot(1,5, 3)
plt.imshow(min_tree_filter.saliencyMapByExtinction(min_tree_attribute, num_leaves), cmap='Grays', vmax=10000, vmin=0)
plt.title('reconstruction (min_tree)')

plt.subplot(1,5, 4)
plt.imshow(tos_filter.saliencyMapByExtinction(tos_attribute, num_leaves), cmap='Grays', vmax=10000, vmin=0)
plt.title('reconstruction (tos)')

plt.subplot(1,5, 5)
plt.imshow(tos_4c8c_filter.saliencyMapByExtinction(tos_4c8c_attribute, num_leaves), cmap='Grays', vmax=10000, vmin=0)
plt.title('reconstruction (tos_4c8c)')
plt.axis('off')

## 5. Filter by direct attribute thresholds

Instead of ranking extrema, this section thresholds a node attribute directly. This is useful when the desired shape or contrast criterion has an interpretable scale.


In [ ]:
threshold = 10000

max_tree_filtered_image = max_tree_filter.filteringMax(max_tree_attribute > threshold)
min_tree_filtered_image = min_tree_filter.filteringMax(min_tree_attribute > threshold)
tos_filtered_image = tos_filter.filteringMax(tos_attribute > threshold)
tos_4c8c_filtered_image = tos_4c8c_filter.filteringMax(tos_4c8c_attribute > threshold)

plt.subplot(1,5, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,5, 2)
plt.imshow(max_tree_filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter (max_tree)')

plt.subplot(1,5, 3)
plt.imshow(min_tree_filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter (min_tree)')

plt.subplot(1,5, 4)
plt.imshow(tos_filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter (tos)')

plt.subplot(1,5, 5)
plt.imshow(tos_4c8c_filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter (tos_4c8c)')